In [1]:
import os
# Disable GPU visibility
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# Disable XLA (critical fix)
os.environ["TF_XLA_FLAGS"] = "--tf_xla_enable_xla_devices=false"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import numpy as np
from PIL import Image
from pathlib import Path
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")
# Disable XLA JIT explicitly
tf.config.optimizer.set_jit(False)

from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

Read YOLO data & process

In [2]:
# DATA PATH
current_dir = Path(os.getcwd())
data_path = current_dir.parent / "Dataset"
if not data_path.exists():
    raise FileNotFoundError(f"Cannot find Dataset. Please check path: {data_path}")

# TODO: For MARS or For MOON
path = os.path.join(data_path,'Mars')
print(path)

train_img_path = os.path.join(path, 'images', 'train')
train_lbl_path = os.path.join(path, 'labels', 'train')

valid_img_path = os.path.join(path, 'images', 'val')
valid_lbl_path = os.path.join(path, 'labels', 'val')

test_img_path = os.path.join(path, 'images', 'test')
test_lbl_path = os.path.join(path, 'labels', 'test')

IMG_SIZE = 128
NUM_CLASSES = 3

# path to save processed images
processed_path = os.path.join(path, 'processed')
os.makedirs(processed_path, exist_ok=True)

/Users/jessie_guo/Downloads/ML Intern - Crater/Dataset/Mars


In [3]:
# process a dataset by extracting craters from images based on YOLO-format labels and saving them in class-specific directories. 
def process_dataset(img_dir, lbl_dir, output_dir):
    for img_file in os.listdir(img_dir):
        if not img_file.endswith('.jpg'):
            continue # skip non-image files
            
        # Get the corresponding label file
        base_name = os.path.splitext(img_file)[0]
        lbl_file = os.path.join(lbl_dir, f"{base_name}.txt")
        
        # Process single image
        img = Image.open(os.path.join(img_dir, img_file))
        img_w, img_h = img.size
        
        with open(lbl_file, 'r') as f:
            for idx, line in enumerate(f.readlines()):
                class_id, xc, yc, w, h = map(float, line.strip().split())
                # Ensure image bounds
                x1 = int((xc - w/2) * img_w)
                y1 = int((yc - h/2) * img_h)
                x2 = int((xc + w/2) * img_w)
                y2 = int((yc + h/2) * img_h)
                                
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(img_w, x2), min(img_h, y2)
                
                # Crop the crater and resize it
                crater = img.crop((x1, y1, x2, y2))
                crater = crater.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
                # save
                class_dir = os.path.join(output_dir, str(int(class_id)))
                os.makedirs(class_dir, exist_ok=True)
                crater.save(os.path.join(class_dir, f"{base_name}_{idx}.jpg"))

# process all dataset
process_dataset(train_img_path, train_lbl_path, os.path.join(processed_path, 'train'))
process_dataset(valid_img_path, valid_lbl_path, os.path.join(processed_path, 'val'))
process_dataset(test_img_path, test_lbl_path, os.path.join(processed_path, 'test'))

In [4]:
# Define data enhancement
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# Loading data from category folder
train_generator = train_datagen.flow_from_directory(
    os.path.join(processed_path, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical'
)

val_generator = train_datagen.flow_from_directory(
    os.path.join(processed_path, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical'
)

# test data
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    os.path.join(processed_path, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical', 
    shuffle=False
)

Found 630 images belonging to 3 classes.
Found 83 images belonging to 3 classes.
Found 171 images belonging to 3 classes.


# Resnet 50

In [6]:
from sklearn.metrics import f1_score as sk_f1

def multi_class_f1(y_true, y_pred):
    def f1_np(y_true_np, y_pred_np):
        y_true_np = np.argmax(y_true_np, axis=1)
        y_pred_np = np.argmax(y_pred_np, axis=1)
        f1 = sk_f1(y_true_np, y_pred_np, average='weighted')
        return np.array(f1, dtype=np.float32)

    f1_score = tf.py_function(func=f1_np, inp=[y_true, y_pred], Tout=tf.float32)
    f1_score.set_shape([])
    return f1_score


In [14]:
ResNet50_model = ResNet50(
    include_top=False,
    weights=None,
    input_shape=(128, 128, 3)
)

# defined output layer
x = layers.GlobalAveragePooling2D()(ResNet50_model.output)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(3, activation='softmax')(x)

model = models.Model(inputs=ResNet50_model.input, outputs=x)

model.compile(
    optimizer = Adam(learning_rate=0.001), 
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint(
    'best_model_resnet_mars.keras', 
    monitor='val_loss', 
    save_best_only=True, 
    mode='min'
)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, mode='min',verbose=0)

history = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=[checkpoint, reduce_lr]
)


Epoch 1/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 88s 2s/step - accuracy: 0.8973 - loss: 0.8696 - val_accuracy: 0.9500 - val_loss: 0.2939 - learning_rate: 0.0010
Epoch 2/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.9260 - loss: 0.4410 - val_accuracy: 0.9500 - val_loss: 0.3540 - learning_rate: 0.0010
Epoch 3/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.9286 - loss: 0.6025 - val_accuracy: 0.9500 - val_loss: 0.3040 - learning_rate: 0.0010
Epoch 4/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.9225 - loss: 0.6115 - val_accuracy: 0.9500 - val_loss: 0.3301 - learning_rate: 0.0010
Epoch 5/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.9417 - loss: 0.3677 - val_accuracy: 0.9500 - val_loss: 0.5103 - learning_rate: 0.0010
Epoch 6/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.9382 - loss: 0.2596 - val_accuracy: 0.9500 - val_loss: 0.5184 - learning_rate: 0.0010
Epoch 7/30
36/36 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.9504 - loss: 0.2426 - val_accuracy: 

KeyboardInterrupt: 

In [8]:
best_model_resnet = load_model('best_model_resnet_mars.keras')
test_loss, test_accuracy = best_model_resnet.evaluate(test_generator)
print(f'\n Test Accuracy: {test_accuracy:.4f}')
print(f' Test loss: {test_loss:.4f}')

21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 221ms/step - accuracy: 0.9106 - loss: 36.8163 

 Test Accuracy: 0.9106
 Test loss: 36.8163


In [9]:
y_pred_probs = best_model_resnet.predict(test_generator)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes 

class_names = list(test_generator.class_indices.keys())

report = classification_report(
    y_true, y_pred_classes,
    target_names=class_names,
    output_dict=True,
    zero_division=1
)

print("\n Per-Class F1-scores:")
for class_name in class_names:
    f1 = report[class_name]['f1-score']
    print(f"  {class_name}: {f1:.4f}")


print("\n Full classification report:")
print(classification_report(y_true, y_pred_classes, target_names=class_names, zero_division=1))


conf_mat = confusion_matrix(y_true, y_pred_classes)
print("\n Confusion Matrix:")
print(conf_mat)

21/21 ━━━━━━━━━━━━━━━━━━━━ 6s 241ms/step

 Per-Class F1-scores:
  0: 0.0000
  1: 0.9532
  2: 0.0000

 Full classification report:
              precision    recall  f1-score   support

           0       1.00      0.00      0.00        15
           1       0.91      1.00      0.95       611
           2       1.00      0.00      0.00        45

    accuracy                           0.91       671
   macro avg       0.97      0.33      0.32       671
weighted avg       0.92      0.91      0.87       671


 Confusion Matrix:
[[  0  15   0]
 [  0 611   0]
 [  0  45   0]]


### 30 times Exp

In [5]:
IMG_SIZE = 128
NUM_CLASSES = 3
NUM_RUNS = 30
class_names = list(test_generator.class_indices.keys())

metrics = {
    'classes': {cls: {'precision': [], 'recall': [], 'f1': []} for cls in class_names},
    'macro_avg': {'precision': [], 'recall': [], 'f1': []},
    'weighted_avg': {'precision': [], 'recall': [], 'f1': []},
    'accuracy': []
}


def create_resnet_model():
    ResNet50_model = ResNet50(
    include_top=False,
    weights=None,
    input_shape=(128, 128, 3)
    )

    # defined output layer
    x = layers.GlobalAveragePooling2D()(ResNet50_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(3, activation='softmax')(x)
    
    model = models.Model(inputs=ResNet50_model.input, outputs=x)
    
    model.compile(
        optimizer = Adam(learning_rate=0.001), 
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


In [6]:
TEMP_MODEL_PATH = 'temp_best_model_resnet_mars.keras'


for run in range(NUM_RUNS):
    print(f"\n=========== Training Run {run + 1}/{NUM_RUNS} ===========")
    
    if os.path.exists(TEMP_MODEL_PATH):
        os.remove(TEMP_MODEL_PATH)

    tf.keras.backend.clear_session()
    np.random.seed(run)
    tf.random.set_seed(run)

    model = create_resnet_model()

    checkpoint = ModelCheckpoint('temp_best_model_resnet_mars.keras', monitor='val_loss', save_best_only=True, mode='min', verbose=0)
    
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, mode='min',verbose=0)
    
    history = model.fit(
        train_generator,
        epochs=30,
        validation_data=val_generator,
        callbacks=[checkpoint, reduce_lr],
        verbose=1
    )

    best_model = load_model('temp_best_model_resnet_mars.keras')

    y_pred = best_model.predict(test_generator, verbose=0)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = test_generator.classes

    report = classification_report(y_true, y_pred_classes,
                                   target_names=class_names,
                                   output_dict=True,
                                   zero_division=1)

    for cls in class_names:
        metrics['classes'][cls]['precision'].append(report[cls]['precision'])
        metrics['classes'][cls]['recall'].append(report[cls]['recall'])
        metrics['classes'][cls]['f1'].append(report[cls]['f1-score'])

    metrics['macro_avg']['precision'].append(report['macro avg']['precision'])
    metrics['macro_avg']['recall'].append(report['macro avg']['recall'])
    metrics['macro_avg']['f1'].append(report['macro avg']['f1-score'])

    metrics['weighted_avg']['precision'].append(report['weighted avg']['precision'])
    metrics['weighted_avg']['recall'].append(report['weighted avg']['recall'])
    metrics['weighted_avg']['f1'].append(report['weighted avg']['f1-score'])

    metrics['accuracy'].append(report['accuracy'])

# PR Curves
precision = dict()
recall = dict()
binarise_y = label_binarize(y_true, classes=[*range(NUM_CLASSES)])

mean_recall = np.linspace(0, 1, 100)
precisions = []

for i in range(NUM_CLASSES):
    precision, recall, _ = precision_recall_curve(binarise_y[:, i],
                                                        y_pred[:, i])
    precisions.append(np.interp(mean_recall, recall[::-1], precision[::-1]))
    plt.plot(recall, precision, lw=2, label='Class {}'.format(i))

mean_precision = np.mean(precisions, axis=0)
plt.plot(mean_recall, mean_precision, lw = 2, label = 'All Classes')
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="best")
plt.title(f"Precision Recall Curves of Renet50 for Mars")
plt.tight_layout()
plt.savefig(f"PR Curves of Renet50 Mars.png")
plt.close()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title('Training Losses of Resnet50 for Mars')
plt.savefig('Training Losses Resnet50 Mars.png')
plt.close()


=========== Training Run 1/30 ===========
Epoch 1/30


I0000 00:00:1769218705.287995 3000913 service.cc:145] XLA service 0x356cd7580 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769218705.288149 3000913 service.cc:153]   StreamExecutor device (0): Host, Default Version
2026-01-24 12:38:25.595009: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769218720.269989 3000913 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


20/20 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.6952 - loss: 1.9016 - val_accuracy: 0.7952 - val_loss: 0.6786 - learning_rate: 0.0010
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.7778 - loss: 1.4993 - val_accuracy: 0.7952 - val_loss: 0.9603 - learning_rate: 0.0010
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.8000 - loss: 1.4150 - val_accuracy: 0.7952 - val_loss: 1.0934 - learning_rate: 0.0010
Epoch 4/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.7571 - loss: 1.2824 - val_accuracy: 0.7952 - val_loss: 1.2604 - learning_rate: 0.0010
Epoch 5/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.7698 - loss: 1.0653 - val_accuracy: 0.7952 - val_loss: 1.1812 - learning_rate: 0.0010
Epoch 6/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.7952 - loss: 0.7249 - val_accuracy: 0.7952 - val_loss: 2.3332 - learning_rate: 0.0010
Epoch 7/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.8397 - loss: 0.4441 - val_accuracy: 0.7952 - va

In [ ]:
print("\n\n=== Classification Report with Mean±Std ===")

class_col_width = max(len(str(cls)) for cls in class_names) + 2
metric_width = 14
support_width = 8

print(f"\n{'Class':<{class_col_width}} {'Precision':<{metric_width}} {'Recall':<{metric_width}} {'F1-score':<{metric_width}} {'Support':<{support_width}}")

for idx, cls in enumerate(class_names):
    prec_mean = np.mean(metrics['classes'][cls]['precision'])
    prec_std = np.std(metrics['classes'][cls]['precision'])
    rec_mean = np.mean(metrics['classes'][cls]['recall'])
    rec_std = np.std(metrics['classes'][cls]['recall'])
    f1_mean = np.mean(metrics['classes'][cls]['f1'])
    f1_std = np.std(metrics['classes'][cls]['f1'])
    support = test_generator.classes.tolist().count(idx)

    print(f"{cls:<{class_col_width}} "
          f"{prec_mean:.2f}±{prec_std:.2f}  "
          f"{rec_mean:.2f}±{rec_std:.2f}  "
          f"{f1_mean:.2f}±{f1_std:.2f}  "
          f"{support:<{support_width}}")

def print_avg_row(name, metric_dict):
    prec = f"{np.mean(metric_dict['precision']):.2f}±{np.std(metric_dict['precision']):.2f}"
    rec = f"{np.mean(metric_dict['recall']):.2f}±{np.std(metric_dict['recall']):.2f}"
    f1 = f"{np.mean(metric_dict['f1']):.2f}±{np.std(metric_dict['f1']):.2f}"
    print(f"{name:<{class_col_width}} {prec:<{metric_width}} {rec:<{metric_width}} {f1:<{metric_width}}")

# accuracy
acc_mean = np.mean(metrics['accuracy'])
acc_std = np.std(metrics['accuracy'])
total_samples = len(test_generator.classes)

print(f"\n{'accuracy':<{class_col_width}} {'':<{metric_width}} {'':<{metric_width}} "
      f"{acc_mean:.2f}±{acc_std:.2f}  {total_samples:<{support_width}}")

print_avg_row('macro avg', metrics['macro_avg'])
print_avg_row('weighted avg', metrics['weighted_avg'])
